# Lab 03 — Build a Deterministic Wordle Environment

**Goal:** build the source of truth that every later evaluation, dataset, and reward depends on.

This lab does **not** use Qwen. That is deliberate. If the environment is subtly wrong, every later benchmark and reward signal is contaminated.

By the end you should be able to:

- score a guess exactly as Wordle does;
- explain why duplicate letters require two-pass scoring;
- represent a game history;
- test candidate words against observed feedback;
- write adversarial tests for the scorer;
- explain why the environment must remain independent of the LLM.

## 3.1 Feedback states

We use:

- `B` — black/gray
- `Y` — yellow
- `G` — green

Strings such as `BYGGB` are easier to store and test than emoji.

In [3]:
from enum import Enum

class Mark(str, Enum):
    GRAY = "B"
    YELLOW = "Y"
    GREEN = "G"

EMOJI = {
    Mark.GRAY: "⬛",
    Mark.YELLOW: "🟨",
    Mark.GREEN: "🟩",
}

def render_feedback(marks):
    return "".join(EMOJI[m] for m in marks)

print("DONE")

DONE


## 3.2 The duplicate-letter trap

The tempting implementation is:

```python
if guess[i] == answer[i]:
    green
elif guess[i] in answer:
    yellow
else:
    gray
```

It is wrong.

Consider:

```text
answer = APPLE
guess  = ALLEY
```

There is only one `L` in `APPLE`, but `ALLEY` guesses `L` twice. Wordle cannot award both occurrences credit.

Correct scoring therefore requires allocation:

1. score greens and consume them;
2. score yellows only from the unmatched letters that remain.

In [5]:
from collections import Counter

def score_guess(answer: str, guess: str) -> tuple[Mark, ...]:
    answer = answer.upper()
    guess = guess.upper()

    if len(answer) != 5 or len(guess) != 5:
        raise ValueError("answer and guess must both contain exactly 5 letters")
    if not answer.isalpha() or not guess.isalpha():
        raise ValueError("answer and guess must contain letters only")

    marks = [Mark.GRAY] * 5
    remaining = Counter()

    # Pass 1: exact-position matches.
    for i, (a, g) in enumerate(zip(answer, guess)):
        if a == g:
            marks[i] = Mark.GREEN
        else:
            remaining[a] += 1

    # Pass 2: misplaced matches, limited by remaining counts.
    for i, g in enumerate(guess):
        if marks[i] == Mark.GREEN:
            continue
        if remaining[g] > 0:
            marks[i] = Mark.YELLOW
            remaining[g] -= 1

    return tuple(marks)

def score_string(answer: str, guess: str) -> str:
    return "".join(mark.value for mark in score_guess(answer, guess))

examples = [
    ("CRANE", "CRANE"),
    ("APPLE", "ALLEY"),
    ("APPLE", "PAPAL"),
    ("BANAL", "LLAMA"),
]

for answer, guess in examples:
    marks = score_guess(answer, guess)
    print(answer, guess, score_string(answer, guess), render_feedback(marks))

CRANE CRANE GGGGG 🟩🟩🟩🟩🟩
APPLE ALLEY GYBYB 🟩🟨⬛🟨⬛
APPLE PAPAL YYGBY 🟨🟨🟩⬛🟨
BANAL LLAMA YBYBY 🟨⬛🟨⬛🟨


### Stop and verify manually

For each example, work through the two passes yourself:

1. Which positions become green?
2. Which answer letters remain unconsumed?
3. Which guessed letters are eligible for yellow?

Do not trust the implementation merely because it ran.

## 3.3 Adversarial tests

The obvious cases are easy. Duplicate letters are where a subtly wrong environment poisons later training.

In [7]:
def check(answer, guess, expected):
    actual = score_string(answer, guess)
    assert actual == expected, (
        f"{answer=} {guess=} expected={expected} actual={actual}"
    )

check("CRANE", "CRANE", "GGGGG")
check("FUZZY", "CRANE", "BBBBB")
check("APPLE", "ALLEY", "GYBYB")
check("CRANE", "MAMMA", "BYBBB")
check("SHEEP", "EERIE", "YYBBB")

print("All scoring tests passed.")

All scoring tests passed.


If an assertion fails, stop and manually verify the expected feedback. Do not "fix" a test merely to make it green.

## 3.4 Represent game history

A Wordle state is accumulated history, not merely the latest row.

In [9]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Turn:
    guess: str
    feedback: str

def play_turn(answer: str, guess: str) -> Turn:
    return Turn(
        guess=guess.upper(),
        feedback=score_string(answer, guess),
    )

history = [
    play_turn("PLANT", "CRANE"),
    play_turn("PLANT", "SLATE"),
]

for turn in history:
    marks = tuple(Mark(ch) for ch in turn.feedback)
    print(turn.guess, turn.feedback, render_feedback(marks))

CRANE BBGGB ⬛⬛🟩🟩⬛
SLATE BGGYB ⬛🟩🟩🟨⬛


The hidden answer is visible only because we are testing the environment. Qwen must never receive it during evaluation. The model gets only guesses and feedback.

## 3.5 Candidate consistency

We *could* immediately build a clever constraint engine tracking greens, forbidden yellow positions, absent letters, and min/max letter counts.

Don't.

For the source of truth, use the scorer we already tested:

> A candidate is consistent if it would have produced exactly the observed feedback for every prior guess.

This avoids maintaining two competing implementations of Wordle semantics.

In [10]:
def is_consistent(candidate: str, history: list[Turn]) -> bool:
    candidate = candidate.upper()
    return all(
        score_string(candidate, turn.guess) == turn.feedback
        for turn in history
    )

candidates = [
    "PLANT",
    "PLANK",
    "PLATE",
    "SLANT",
    "GRANT",
    "CRANE",
]

for candidate in candidates:
    print(f"{candidate}: {is_consistent(candidate, history)}")

PLANT: True
PLANK: False
PLATE: False
SLANT: False
GRANT: False
CRANE: False


In [11]:
def filter_candidates(words: list[str], history: list[Turn]) -> list[str]:
    return [
        word.upper()
        for word in words
        if len(word) == 5
        and word.isalpha()
        and is_consistent(word, history)
    ]

toy_dictionary = [
    "PLANT", "PLANK", "PLATE", "SLANT", "GRANT",
    "CRANE", "POINT", "BLIND", "CHANT",
]

print(filter_candidates(toy_dictionary, history))

['PLANT']


## 3.6 Connect back to Lab 01

Qwen struggled to translate feedback into constraints. Our environment can now generate those states exactly.

Representation remains a separate experimental variable.

In [12]:
def format_history(history: list[Turn], spaced_letters: bool = False) -> str:
    lines = []
    for turn in history:
        guess = " ".join(turn.guess) if spaced_letters else turn.guess
        feedback = " ".join(turn.feedback) if spaced_letters else turn.feedback
        lines.append(f"{guess} -> {feedback}")
    return "\n".join(lines)

print("NORMAL")
print(format_history(history))

print("\nSPACED")
print(format_history(history, spaced_letters=True))

NORMAL
CRANE -> BBGGB
SLATE -> BGGYB

SPACED
C R A N E -> B B G G B
S L A T E -> B G G Y B


Later we can compare:

```text
CRANE -> BBYGB
```

with:

```text
C R A N E -> B B Y G B
```

Same game state. Same model. Same training data. Only representation changes.

## 3.7 Exhaustive toy sanity check

In [18]:
toy_words = [
    "CRANE", "SLATE", "AUDIO", "POINT",
    "SHORE", "BLIND", "MIGHT", "ROUND",
    "CHAMP", "FLING", "BRICK", "GHOST",
    "PLUMB", "WASTE", "KNIFE", "DOUBT",
    "APPLE", "SHEEP", "BANAL", "ALLEY",
    "PLANT",
]

pairs_checked = 0

for answer in toy_words:
    for guess in toy_words:
        result = score_guess(answer, guess)
        assert len(result) == 5
        assert all(isinstance(mark, Mark) for mark in result)

        if answer == guess:
            assert result == (Mark.GREEN,) * 5

        pairs_checked += 1

print(f"Checked {pairs_checked} answer/guess pairs.")

Checked 441 answer/guess pairs.


## 3.8 Promote infrastructure into `src/`

Notebook code is exploratory. Stable infrastructure belongs in the package.

This cell assumes the notebook is running from the repo's `notebooks/` directory and writes `../src/tiny_wordle/game.py`.

In [14]:
from pathlib import Path

game_py = Path("../src/tiny_wordle/game.py")
game_py.parent.mkdir(parents=True, exist_ok=True)

game_source = '''from __future__ import annotations

from collections import Counter
from dataclasses import dataclass
from enum import Enum


class Mark(str, Enum):
    GRAY = "B"
    YELLOW = "Y"
    GREEN = "G"


@dataclass(frozen=True)
class Turn:
    guess: str
    feedback: str


def score_guess(answer: str, guess: str) -> tuple[Mark, ...]:
    answer = answer.upper()
    guess = guess.upper()

    if len(answer) != 5 or len(guess) != 5:
        raise ValueError("answer and guess must both contain exactly 5 letters")
    if not answer.isalpha() or not guess.isalpha():
        raise ValueError("answer and guess must contain letters only")

    marks = [Mark.GRAY] * 5
    remaining = Counter()

    for i, (a, g) in enumerate(zip(answer, guess)):
        if a == g:
            marks[i] = Mark.GREEN
        else:
            remaining[a] += 1

    for i, g in enumerate(guess):
        if marks[i] == Mark.GREEN:
            continue
        if remaining[g] > 0:
            marks[i] = Mark.YELLOW
            remaining[g] -= 1

    return tuple(marks)


def score_string(answer: str, guess: str) -> str:
    return "".join(mark.value for mark in score_guess(answer, guess))


def play_turn(answer: str, guess: str) -> Turn:
    return Turn(guess=guess.upper(), feedback=score_string(answer, guess))


def is_consistent(candidate: str, history: list[Turn]) -> bool:
    candidate = candidate.upper()
    return all(
        score_string(candidate, turn.guess) == turn.feedback
        for turn in history
    )


def filter_candidates(words: list[str], history: list[Turn]) -> list[str]:
    return [
        word.upper()
        for word in words
        if len(word) == 5
        and word.isalpha()
        and is_consistent(word, history)
    ]
'''

game_py.write_text(game_source)
print(f"Wrote {game_py}")

Wrote ../src/tiny_wordle/game.py


## 3.9 Real unit tests

Now make the scorer independently testable.

In [15]:
test_py = Path("../tests/test_game.py")
test_py.parent.mkdir(parents=True, exist_ok=True)

test_source = '''import pytest

from tiny_wordle.game import Turn, filter_candidates, is_consistent, score_string


@pytest.mark.parametrize(
    "answer,guess,expected",
    [
        ("CRANE", "CRANE", "GGGGG"),
        ("FUZZY", "CRANE", "BBBBB"),
        ("APPLE", "ALLEY", "GYBYB"),
        ("CRANE", "MAMMA", "BYBBB"),
        ("SHEEP", "EERIE", "YYBBB"),
    ],
)
def test_score_guess(answer, guess, expected):
    assert score_string(answer, guess) == expected


def test_invalid_length():
    with pytest.raises(ValueError):
        score_string("CRANE", "CAT")


def test_consistency_replays_feedback():
    history = [
        Turn("CRANE", score_string("PLANT", "CRANE")),
        Turn("SLATE", score_string("PLANT", "SLATE")),
    ]
    assert is_consistent("PLANT", history)
    assert not is_consistent("CRANE", history)


def test_filter_candidates():
    history = [Turn("CRANE", score_string("PLANT", "CRANE"))]
    words = ["PLANT", "CRANE", "POINT", "CHANT"]
    result = filter_candidates(words, history)

    assert "PLANT" in result
    assert "CRANE" not in result
'''

test_py.write_text(test_source)
print(f"Wrote {test_py}")

Wrote ../tests/test_game.py


In [16]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pytest", "../tests/test_game.py", "-q"],
    capture_output=True,
    text=True,
)

print(result.stdout)
print(result.stderr)
assert result.returncode == 0

........                                                                 [100%]
8 passed in 0.01s




## 3.10 Manual game experiment

Choose a hidden answer and three guesses.

Before running the cell, predict each feedback row yourself.

In [19]:
answer = "PLANT"
guesses = ["CRANE", "SLATE", "POINT"]

manual_history = []

for guess in guesses:
    turn = play_turn(answer, guess)
    manual_history.append(turn)

    print(
        f"{guess} -> {turn.feedback} | "
        f"remaining toy candidates: "
        f"{filter_candidates(toy_words, manual_history)}"
    )

CRANE -> BBGGB | remaining toy candidates: ['PLANT']
SLATE -> BGGYB | remaining toy candidates: ['PLANT']
POINT -> GBBGG | remaining toy candidates: ['PLANT']


## Lab 03 checkpoint

You should now be able to explain:

1. Why naive `letter in answer` scoring is wrong.
2. Why greens must be allocated before yellows.
3. Why duplicate letters contain count information.
4. Why candidate consistency can safely reuse the scorer.
5. Why the Wordle environment must not use an LLM.
6. Why the same environment can drive evaluation, synthetic data, and later RL rewards.
7. Why representation belongs outside the environment.

### Send me these results

- the four rows from the example scoring cell;
- `All scoring tests passed.`;
- the candidate-consistency output;
- the pytest result;
- the three-turn manual game output.

### Next

**Lab 04 — Baseline Evaluation**

We'll let the untouched Qwen3-0.6B play real games against held-out answers and quantify the failures we were eyeballing in Lab 01.